Прежде чем сдать эту работу, убедитесь, что всё выполняется так, как ожидается.
Сначала перезапустите ядро (в меню выберите Kernel → Restart), а затем запустите все ячейки (в меню выберите Cell → Run All).

Убедитесь, что вы заполнили все места, где указано YOUR CODE HERE или "YOUR ANSWER HERE", а также указали своё имя ниже.

In [ ]:
NAME = ""

Убедитесь, что используете версию Python > 3. Все тетрадки отрабатывают точно на версии 3.10.10.

In [ ]:
import IPython
assert IPython.version_info[0] >= 3, "Your version of IPython is too old, please update it."

---

## Домашнее задание 3. Классификация текста

### О задании

В этом задании вы познакомитесь с основами классификации текста. Вам предстоит:

1. Научиться очищать и предобрабатывать текст
2. Построить различные признаковые представления (Bag-of-Words, TF-IDF, Word2Vec)
3. Разобраться в метриках классификации (Precision, Recall, F1, ROC-AUC, PR-AUC)
4. Обучить и настроить классические модели: логистическую регрессию, SVM, KNN
5. Провести эксперименты на разных типах признаков и сравнить результаты
6. Проинтерпретировать результаты моделей

Выполнив этот ноутбук можно получить максимум 9 баллов. 

Оценку 10 могут получить те, кто выполнил текущую ДЗ на 9 баллов и вошел в первые 10 человек потока, набравшие максимальное значение F1-метрики на тесте. Для этого необходимо в конце ДЗ прислать свое ФИО и результат на тесте в [форму](https://forms.gle/Sk3NbUzNi4rqDpLj8). При этом если первые 9 человек имеют уникальное значение метрики, а у 10-го и 11-го оценки совпадают, в таком случае оценка 10 ставится первым 11 студентам.

In [ ]:
!pip3 install sentence-transformers pymorphy3 nltk

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, auc,
)
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.dummy import DummyClassifier

from sentence_transformers import SentenceTransformer

import nltk
from nltk.corpus import stopwords
from pymorphy3 import MorphAnalyzer

nltk.download('stopwords', quiet=True)

morph = MorphAnalyzer()

warnings.filterwarnings('ignore')
%matplotlib inline

### Загрузка данных

Мы будем работать с датасетом **CoAT** (Corpus of Artificial Texts) - корпусом русскоязычных текстов.
Каждый текст написан либо человеком, либо одной из 13 нейросетей (OPUS-MT, M2M-100, M-BART50, ruGPT3 и др.).
Задача - по тексту определить его автора.

Будем использовать подвыборку из 10 000 примеров для ускорения работы.

In [ ]:
df = pd.read_csv('coat_data.csv')

print(f'Размер датасета: {df.shape}')
print(f'Классы: {sorted(df["label"].unique())}')
df.head()

### Разведочный анализ

In [ ]:
print('Распределение классов:')
print(df['label'].value_counts())
print(f'\nВсего классов: {df["label"].nunique()}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
df['label'].value_counts().plot.bar(ax=ax)
ax.set_title('Распределение классов в датасете CoAT')
ax.set_xlabel('Автор (Human / нейросеть)')
ax.set_ylabel('Количество текстов')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Обратите внимание на дисбаланс классов: текстов, написанных людьми, значительно больше, чем текстов каждой отдельной нейросети. Это важно учитывать при выборе метрики оценки качества

Закодируем классы и разделим выборку на трейн и тест:

In [ ]:
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'], df['label_encoded'],
    test_size=0.2, random_state=42, stratify=df['label_encoded']
)

print(f'Обучающая выборка: {len(train_texts)}')
print(f'Тестовая выборка: {len(test_texts)}')
print(f'\nКлассы: {list(le.classes_)}')

---
## Секция 1. Очистка текста (1 балл)

Перед тем как строить признаки из текста, его необходимо очистить. Основные шаги предобработки:

1. Приведение к нижнему регистру - чтобы «Кот» и «кот» считались одним словом, а не двумя разными токенами.

2. Удаление пунктуации и цифр - знаки препинания и числа обычно не несут полезной информации для классификации тематики текста. Например, строка "Цена: 1500 руб.!" после очистки превратится в `"цена руб"`.

3. Удаление стоп-слов - это высокочастотные слова (`и`, `в`, `на`, `что`, `это`, `не`...), которые встречаются почти в каждом тексте и не помогают отличить один класс от другого. Библиотека NLTK предоставляет готовый список через `nltk.corpus.stopwords.words('russian')`.

4. Стемминг и лемматизация - приведение слов к базовой форме, чтобы разные словоформы одного слова объединялись в один признак.

**Стемминг** - грубо отсекает окончания и суффиксы по правилам. Работает быстро, но может ошибаться:

| Слово | Стем (Snowball) |
|-------|----------------|
| бежал | бежа |
| бегу | бег |
| бежать | бежа |
| красивый | красив |
| красота | красот |

Как видно, «бегу» и «бежал» получают разные стемы - стеммер не знает, что это формы одного глагола.

**Лемматизация** - использует морфологический словарь и возвращает нормальную форму (лемму):

| Слово | Лемма |
|-------|-------------------|
| бежал | бежать |
| бегу | бежать |
| бежать | бежать |
| красивый | красивый |
| красота | красота |

Все формы глагола «бежать» корректно сводятся к одной лемме.

Почему для русского языка лемматизация предпочтительнее? Русский - морфологически богатый язык: одно слово может иметь десятки форм (падежи, числа, рода, времена, залоги). Стеммер, который просто обрезает окончания, часто ошибается: объединяет разные слова (`лес` и `леса` - стем `лес`, но `лесть` тоже даст `лес`) или, наоборот, не объединяет формы одного слова. Лемматизатор, опираясь на словарь, работает точнее, хотя и медленнее.

В этом задании вы реализуете первые три шага, а лемматизацию применим отдельно с помощью библиотеки `pymorphy3`.

#### Задание 1.1. Очистка текста (0.3 балла)

Реализуйте функцию `clean_text`, которая:
1. Приводит текст к нижнему регистру
2. Удаляет всё, кроме кириллических и латинских букв и пробелов
3. Заменяет множественные пробелы на один
4. Убирает пробелы в начале и конце строки

In [ ]:
def clean_text(text: str) -> str:
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
%%capture

assert clean_text("Привет, МИР! 123") == "привет мир"
assert clean_text("  Много   пробелов  ") == "много пробелов"
assert clean_text("Hello World!") == "hello world"
assert clean_text("Тест123тест") == "тесттест"

#### Задание 1.2. Удаление стоп-слов (0.3 балла)

Реализуйте функцию `remove_stopwords`, которая принимает список токенов и множество стоп-слов, и возвращает отфильтрованный список.

Для русского языка удобно использовать `nltk.corpus.stopwords.words('russian')`.

In [ ]:
def remove_stopwords(tokens: list, stop_words: set) -> list:
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
%%capture

russian_sw = set(stopwords.words('russian'))
tokens = ["это", "кот", "и", "собака", "в", "парке"]
result = remove_stopwords(tokens, russian_sw)
assert "и" not in result
assert "в" not in result
assert "кот" in result
assert "собака" in result
assert "парке" in result

#### Задание 1.3. Предобработка корпуса (0.4 балла)

Реализуйте функцию `preprocess_corpus`, которая применяет полный pipeline предобработки ко всей серии текстов:
1. `clean_text` - очистка
2. `word_tokenize` - токенизация
3. `remove_stopwords` - удаление стоп-слов
4. Объединение токенов обратно в строку через пробел

Функция должна вернуть `pd.Series` той же длины, что и входная.

In [ ]:
def word_tokenize(text: str) -> list:
    return re.findall(r'[а-яёА-ЯЁa-zA-Z]+', text)

In [ ]:
def preprocess_corpus(texts: pd.Series) -> pd.Series:
    """Применить полный pipeline очистки ко всему корпусу."""
    russian_sw = set(stopwords.words('russian'))
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
%%capture

test_series = pd.Series(["Это ТЕСТ, 123!", "Привет Мир и Всё"])
result = preprocess_corpus(test_series)
assert isinstance(result, pd.Series), "Результат должен быть pd.Series"
assert len(result) == 2
for text in result:
    assert text == text.lower(), "Текст должен быть в нижнем регистре"
    assert not any(c.isdigit() for c in text), "Цифр быть не должно"

Применим предобработку к нашим данным:

In [ ]:
train_texts_clean = preprocess_corpus(train_texts.reset_index(drop=True))
test_texts_clean = preprocess_corpus(test_texts.reset_index(drop=True))

print('Пример до очистки:')
print(train_texts.iloc[0][:200], '...')
print('\nПосле очистки:')
print(train_texts_clean.iloc[0][:200], '...')

### Лемматизация

Теперь применим лемматизацию с помощью `pymorphy3`. Для каждого слова морфологический анализатор определит его нормальную форму (лемму). Например:
- «бежал», «бегу», «бежать» → бежать
- «красивого», «красивым» → красивый

Это позволит объединить разные словоформы в один признак и уменьшить размерность словаря.

In [ ]:
def lemmatize_text(text: str) -> str:
    tokens = text.split()
    lemmas = [morph.parse(token)[0].normal_form for token in tokens]
    return ' '.join(lemmas)

train_texts_clean = train_texts_clean.apply(lemmatize_text)
test_texts_clean = test_texts_clean.apply(lemmatize_text)

print('Пример после лемматизации:')
print(train_texts_clean.iloc[0][:200], '...')

---
## Секция 2. Векторизация текста (1.5 балла)

Модели машинного обучения работают с числовыми признаками, а не с сырым текстом. Необходимо преобразовать тексты в числовые векторы. Мы рассмотрим три подхода.

### Bag-of-Words (CountVectorizer)
Каждый текст представляется вектором, где каждый элемент - количество вхождений слова из словаря. Порядок слов теряется (отсюда название «мешок слов»). Результат - разреженная матрица размера `(n_документов, n_слов_в_словаре)`.

### TF-IDF (Term Frequency - Inverse Document Frequency)

Улучшение BoW: слова взвешиваются с учётом их «важности». Идея состоит из двух множителей:

**TF (Term Frequency)** - как часто слово $t$ встречается в документе $d$:

$$\text{tf}(t, d) = \frac{\text{число вхождений } t \text{ в } d}{\text{общее число слов в } d}$$

**IDF (Inverse Document Frequency)** - обратная документная частота, штрафует слова, которые встречаются во многих документах (предлоги, союзы и т.д.):

$$\text{idf}(t, D) = \log \frac{|D|}{|\{d \in D : t \in d\}|}$$

где $|D|$ - общее число документов в корпусе, а $|\{d \in D : t \in d\}|$ - число документов, содержащих слово $t$.

Итоговый вес слова:

$$\text{tf-idf}(t, d, D) = \text{tf}(t, d) \cdot \text{idf}(t, D)$$

Слова, частые в конкретном документе, но редкие в корпусе, получают высокий вес - они наиболее характерны для этого документа. Слова, встречающиеся повсеместно (например, «и», «в», «на»), получают низкий IDF и, как следствие, низкий итоговый вес.

TF-IDF можно считать на уровне слов или n-грамм - последнее позволяет учитывать морфологические паттерны.

### Sentence Embeddings (SentenceTransformer)
Современный подход: каждый текст целиком кодируется предобученной трансформером в плотный вектор фиксированной размерности. В отличие от BoW/TF-IDF, эмбеддинги учитывают контекст и семантику: тексты с похожим смыслом получают близкие векторы, даже если используют разные слова. Мы будем использовать библиотеку `sentence-transformers`.

#### Задание 2.1. Bag-of-Words (0.5 балла)

Реализуйте функцию `build_bow_features`, которая с помощью `CountVectorizer` строит Bag-of-Words признаки.

Функция должна вернуть кортеж `(X_train, X_test, vectorizer)`.

In [ ]:
def build_bow_features(train_texts, test_texts):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
%%capture

_X_tr, _X_te, _vec = build_bow_features(train_texts_clean, test_texts_clean)
assert _X_tr.shape[0] == len(train_texts_clean)
assert _X_te.shape[0] == len(test_texts_clean)
assert _X_tr.shape[1] == _X_te.shape[1]
assert _X_tr.shape[1] > 100, "Словарь слишком маленький"
del _X_tr, _X_te, _vec

#### Задание 2.2. TF-IDF (0.5 балла)

Реализуйте функцию `build_tfidf_features`, которая строит TF-IDF признаки. Используйте [документацию](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html). 

Функция должна вернуть `(X_train, X_test, vectorizer)`.

In [ ]:
def build_tfidf_features(train_texts, test_texts, analyzer='word', ngram_range=(2, 4)):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
%%capture

_Xw_tr, _Xw_te, _vw = build_tfidf_features(train_texts_clean, test_texts_clean, analyzer='word')
_Xc_tr, _Xc_te, _vc = build_tfidf_features(train_texts_clean, test_texts_clean, analyzer='char')
assert _Xw_tr.shape[0] == len(train_texts_clean)
assert _Xc_tr.shape[1] != _Xw_tr.shape[1], "word и char должны давать разное число признаков"
del _Xw_tr, _Xw_te, _vw, _Xc_tr, _Xc_te, _vc

#### Задание 2.3. Sentence Embeddings (0.5 балла)

Реализуйте функцию `build_embedding_features`, которая:

1. Загружает предобученную модель `SentenceTransformer` (по умолчанию `'all-MiniLM-L6-v2'`, но вы можете использовать другую модель)
2. Кодирует тексты в плотные векторы с помощью `model.encode()`
3. Возвращает `(X_train, X_test, model)` - numpy-матрицы и модель

`encode()` принимает список строк и возвращает numpy-массив. Используйте `show_progress_bar=True` для отслеживания прогресса.

In [ ]:
def build_embedding_features(train_texts, test_texts, model_name='all-MiniLM-L6-v2'):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
%%capture

_Xe_tr, _Xe_te, _st_model = build_embedding_features(train_texts_clean, test_texts_clean)
assert _Xe_tr.shape[0] == len(train_texts_clean), f"Ожидается {len(train_texts_clean)} строк, получено {_Xe_tr.shape[0]}"
assert _Xe_te.shape[0] == len(test_texts_clean)
assert _Xe_tr.shape[1] == _Xe_te.shape[1], "Размерности train и test должны совпадать"
assert _Xe_tr.shape[1] > 10, "Размерность эмбеддингов слишком маленькая"
assert np.all(np.isfinite(_Xe_tr)), "Векторы содержат NaN/Inf"
del _Xe_tr, _Xe_te, _st_model

Построим все типы признаков для дальнейших экспериментов:

In [ ]:
print('Строим все типы признаков...\n')

X_tr_bow, X_te_bow, bow_vec = build_bow_features(train_texts_clean, test_texts_clean)
print(f'BoW: {X_tr_bow.shape[1]} признаков')

X_tr_tfidf_w, X_te_tfidf_w, tfidf_w_vec = build_tfidf_features(train_texts_clean, test_texts_clean, analyzer='word')
print(f'TF-IDF (word): {X_tr_tfidf_w.shape[1]} признаков')

X_tr_tfidf_c, X_te_tfidf_c, tfidf_c_vec = build_tfidf_features(train_texts_clean, test_texts_clean, analyzer='char')
print(f'TF-IDF (char): {X_tr_tfidf_c.shape[1]} признаков')

X_tr_emb, X_te_emb, st_model = build_embedding_features(train_texts_clean, test_texts_clean)
print(f'Sentence Embeddings: {X_tr_emb.shape[1]} признаков')

features = {
    'BoW': (X_tr_bow, X_te_bow),
    'TF-IDF (word)': (X_tr_tfidf_w, X_te_tfidf_w),
    'TF-IDF (char)': (X_tr_tfidf_c, X_te_tfidf_c),
    'Embeddings': (X_tr_emb, X_te_emb),
}

print(f'\nВсего типов признаков: {len(features)}')

---
## Секция 3. Метрики классификации (1.0 балл)

Прежде чем обучать модели, необходимо разобраться, как оценивать их качество.

### Матрица ошибок (Confusion Matrix)

Для бинарной классификации матрица ошибок выглядит так:

| | Предсказано: Positive | Предсказано: Negative |
|---|---|---|
| **Реально: Positive** | TP (True Positive) | FN (False Negative) |
| **Реально: Negative** | FP (False Positive) | TN (True Negative) |

- **TP** - верно предсказанные положительные
- **TN** - верно предсказанные отрицательные
- **FP** - ложные срабатывания (ошибка I рода)
- **FN** - пропуски (ошибка II рода)

Из этих четырёх величин строятся все основные метрики.

#### Задание 3.1. Формула Precision (0.15 балла)
Выпишите **в следующей ячйке** формулу **Precision** (точность) через элементы confusion matrix: TP и FP.

YOUR ANSWER HERE

#### Задание 3.2. Формула Recall (0.15 балла)
Выпишите **в следующей ячейке** формулу **Recall** (полноту) через элементы confusion matrix: TP и FN.

YOUR ANSWER HERE

#### Задание 3.3. Формула Accuracy (0.15 балла)
Выпишите **в следующей ячейке** формулу **Accuracy** через TP, TN, FP и FN.

YOUR ANSWER HERE

#### Задание 3.4. Формула F1-score (0.15 балла)
Выпишите **в следующей ячейке** формулу **F1-score** через Precision и Recall.

YOUR ANSWER HERE

### ROC-AUC и PR-AUC

Помимо точечных метрик (Precision, Recall, F1), существуют метрики, оценивающие качество ранжирования по всем порогам:

**ROC-AUC** (Area Under ROC Curve):
ROC-кривая строится как зависимость TPR (Recall) от FPR (False Positive Rate = FP / (FP + TN))

**PR-AUC** (Area Under Precision-Recall Curve):
PR-кривая строится как зависимость Precision от Recall

#### Задание 3.5. ROC-AUC vs PR-AUC (0.4 балла)
Ответьте **в следующей ячейке** на следующие вопросы:
1. Когда лучше использовать ROC-AUC, а когда PR-AUC?
2. Как наличие сильного дисбаланса классов влияет на информативность ROC-AUC?
3. Какую из этих двух метрик вы бы выбрали для нашей задачи (CoAT, определение авторства) и почему?

YOUR ANSWER HERE

### Выбор метрики для этого задания

Наш датасет **несбалансирован**: текстов от человека значительно больше, чем от каждой отдельной нейросети.

**Accuracy** здесь будет обманчива: модель, которая всегда предсказывает класс «Human», получит высокую accuracy, но будет бесполезна для распознавания конкретных нейросетей.

Поэтому основная метрика - **F1-macro**:
$$F1_{\text{macro}} = \frac{1}{K} \sum_{k=1}^{K} F1_k$$

Она усредняет F1 по всем классам с равным весом, давая одинаковую важность как частым, так и редким классам.

---
### Базовая модель (DummyClassifier)

Прежде чем обучать сложные модели, обучим baseline. `DummyClassifier` со стратегией `stratified` предсказывает классы случайно, пропорционально их частотам в обучающей выборке. Любая обученная модель должна быть значительно лучше.

In [ ]:
dummy = DummyClassifier(strategy='stratified', random_state=42)
dummy.fit(features['BoW'][0], train_labels)
dummy_pred = dummy.predict(features['BoW'][1])
dummy_f1 = f1_score(test_labels, dummy_pred, average='macro')
dummy_acc = accuracy_score(test_labels, dummy_pred)

print(f'DummyClassifier (stratified):')
print(f'  F1-macro:  {dummy_f1:.4f}')
print(f'  Accuracy:  {dummy_acc:.4f}')

---
## Секция 4. Логистическая регрессия (1.5 балла)

### Напоминание

Логистическая регрессия - один из самых популярных линейных классификаторов. Несмотря на слово «регрессия» в названии, это метод классификации.

Для бинарного случая модель предсказывает вероятность с помощью сигмоиды:
$$P(y=1 | \mathbf{x}) = \sigma(\mathbf{w}^T \mathbf{x}) = \frac{1}{1 + e^{-\mathbf{w}^T \mathbf{x}}}$$

Для мультикласса используется обобщение - oftmax:
$$P(y=k | \mathbf{x}) = \frac{e^{\mathbf{w}_k^T \mathbf{x}}}{\sum_{j=1}^{K} e^{\mathbf{w}_j^T \mathbf{x}}}$$

Параметры обучаются минимизацией log loss.

#### Задание 4.1. Формула Log Loss (0.25 балла)
Выпишите **в следующей ячейке** формулу Log Loss для бинарной классификации. Объясните, что происходит, когда модель уверенно предсказывает неправильный класс.

YOUR ANSWER HERE

#### Задание 4.2. Обучение логистической регрессии (1.25 балл)

Реализуйте функцию `train_logreg`, которая перебирает гиперпараметры `LogisticRegression` с помощью кросс-валидации. Переберите параметр `C`, а также самостоятельно загляните в [документацию](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) и подумайте, что ещё можно перебрать

Используйте `GridSearchCV` с `scoring='f1_macro'`

In [ ]:
def train_logreg(X_train, y_train):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
%%capture

_test_gs = train_logreg(features['BoW'][0], train_labels)
assert hasattr(_test_gs, 'best_score_'), "Должен быть обученный GridSearchCV"
assert hasattr(_test_gs, 'best_params_'), "GridSearchCV должен иметь best_params_"
assert _test_gs.best_score_ > dummy_f1, f"LogReg должен быть лучше Dummy ({dummy_f1:.4f})"
assert len(_test_gs.cv_results_['params']) > 1, "Сетка должна содержать больше одной комбинации параметров"
del _test_gs

In [ ]:
print('Обучаем LogReg на всех типах признаков...\n')

logreg_results = {}
for name, (X_tr, X_te) in features.items():
    print(f'  {name}...', end=' ', flush=True)
    gs = train_logreg(X_tr, train_labels)
    logreg_results[name] = gs
    print(f'best CV F1-macro: {gs.best_score_:.4f}, params: {gs.best_params_}')

#### ROC и PR кривые (0 балла)

Давайте для каждого класса визуализируем ROC-кривую и PR-кривую:

In [ ]:
def plot_roc_pr_curves(model, X_test, y_test, class_names=None):
    est = model.best_estimator_ if hasattr(model, 'best_estimator_') else model
    y_score = est.predict_proba(X_test)

    classes = sorted(np.unique(y_test))
    y_test_bin = label_binarize(y_test, classes=classes)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    for i, cls in enumerate(classes):
        name = class_names[i] if class_names is not None else f'Class {cls}'

        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc_val = auc(fpr, tpr)
        axes[0].plot(fpr, tpr, label=f'{name} ({roc_auc_val:.2f})', alpha=0.7)

        prec, rec, _ = precision_recall_curve(y_test_bin[:, i], y_score[:, i])
        pr_auc_val = auc(rec, prec)
        axes[1].plot(rec, prec, label=f'{name} ({pr_auc_val:.2f})', alpha=0.7)

    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
    axes[0].set_xlabel('FPR')
    axes[0].set_ylabel('TPR')
    axes[0].set_title('ROC Curves (One-vs-Rest)')
    axes[0].legend(fontsize=7, loc='lower right')
    axes[0].grid(alpha=0.3)

    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title('PR Curves (One-vs-Rest)')
    axes[1].legend(fontsize=7, loc='lower left')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
best_lr_feat = max(logreg_results, key=lambda k: logreg_results[k].best_score_)
plot_roc_pr_curves(logreg_results[best_lr_feat], features[best_lr_feat][1], test_labels, class_names=le.classes_)

---
## Секция 5. SVM (1.0 балл)

### Напоминание

**SVM (Support Vector Machine)** - метод классификации, который ищет разделяющую гиперплоскость с максимальным **отступом** (margin) между классами.

#### Задание 5.1. Обучение SVM (1.0 балл)

Реализуйте функцию `train_svm`, которая перебирает гиперпараметры для `SVC` с помощтю кросс-валидации. Аналогично предыдущему заданию, используйте `GridSearchCV` с `scoring='f1_macro'`.

Обязательно переберите параметр `C`, но также гляньте в документации другие параметры, которые можно перебрать

In [ ]:
def train_svm(X_train, y_train) -> GridSearchCV:
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
%%capture

_test_gs = train_svm(features['BoW'][0], train_labels)
assert hasattr(_test_gs, 'best_score_')
assert hasattr(_test_gs, 'best_params_')
assert _test_gs.best_score_ > dummy_f1, f"SVM должен быть лучше Dummy ({dummy_f1:.4f})"
assert len(_test_gs.cv_results_['params']) > 1, "Сетка должна содержать больше одной комбинации"
del _test_gs

In [ ]:
print('Обучаем SVM на всех типах признаков...\n')

svm_results = {}
for name, (X_tr, X_te) in features.items():
    print(f'  {name}...', end=' ', flush=True)
    gs = train_svm(X_tr, train_labels)
    svm_results[name] = gs
    print(f'best CV F1-macro: {gs.best_score_:.4f}, params: {gs.best_params_}')

---
## Секция 6. KNN (1.0 балл)

### Напоминание

**KNN (K Nearest Neighbors)** - метод классификации, основанный на «ближайших соседях». Для нового объекта находятся $k$ ближайших объектов из обучающей выборки, и класс определяется голосованием большинства.

#### Задание 6.1. Обучение KNN (1 балл)

Реализуйте функцию `train_knn`, которая также как и в предыдущих заданиях перебирает гиперпараметры для `KNeighborsClassifier`

In [ ]:
def train_knn(X_train, y_train) -> GridSearchCV:
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
%%capture

_test_gs = train_knn(features['BoW'][0], train_labels)
assert hasattr(_test_gs, 'best_score_')
assert hasattr(_test_gs, 'best_params_')
assert len(_test_gs.cv_results_['params']) > 1, "Сетка должна содержать больше одной комбинации"
del _test_gs

In [ ]:
print('Обучаем KNN на всех типах признаков...\n')

knn_results = {}
for name, (X_tr, X_te) in features.items():
    print(f'  {name}...', end=' ', flush=True)
    gs = train_knn(X_tr, train_labels)
    knn_results[name] = gs
    print(f'best CV F1-macro: {gs.best_score_:.4f}, params: {gs.best_params_}')

---
## Секция 7. Сравнение моделей и выводы (2.0 балла)

Мы обучили три модели (LogReg, SVM, KNN) на четырёх типах признаков (BoW, TF-IDF word, TF-IDF char, Embeddings). Теперь нужно свести все результаты в одну таблицу, сделать выводы и проинтерпретировать лучшую модель.

#### Задание 7.1. Сравнительная таблица (0.5 балла)

Реализуйте функцию `compare_models`, которая:
1. Для каждой комбинации (модель × тип признаков) делает предсказание на тестовой выборке
2. Вычисляет F1-macro и Accuracy
3. Добавляет строку с DummyClassifier для сравнения
4. Возвращает `pd.DataFrame`, отсортированный по F1-macro

DataFrame должен содержать колонки: `Model`, `Features`, `F1-macro`, `Accuracy`

In [ ]:
def compare_models(
    all_results: dict[str, dict[str, GridSearchCV]], 
    features: dict[str, tuple[np.ndarray, np.ndarray]], 
    y_test: np.ndarray, 
    dummy_f1: float, 
    dummy_acc: float
) -> pd.DataFrame:
    """Сравнить все модели на всех типах признаков.

    Args:
        all_results: {'LogReg': {feat_name: gs, ...}, 'SVM': {...}, 'KNN': {...}}
        features: {feat_name: (X_tr, X_te)}
        y_test: тестовые метки
        dummy_f1, dummy_acc: метрики DummyClassifier
    Returns:
        pd.DataFrame sorted by F1-macro
    """
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
%%capture

all_results = {
    'LogReg': logreg_results,
    'SVM': svm_results,
    'KNN': knn_results,
}

comparison_df = compare_models(all_results, features, test_labels, dummy_f1, dummy_acc)
assert isinstance(comparison_df, pd.DataFrame)
assert 'F1-macro' in comparison_df.columns
assert len(comparison_df) >= 13, f"Ожидается >= 13 строк (3 модели × 4 признака + dummy), получено {len(comparison_df)}"

In [ ]:
comparison_df

#### Задание 7.2. Выводы по экспериментам (0.75 балла)
На основе таблицы сравнения ответьте на следующие вопросы **в следующей ячейке**:

1. Какая комбинация **модель + тип признаков** показала лучший результат? Почему, на ваш взгляд?
2. Какой **тип признаков** (BoW, TF-IDF word, TF-IDF char, Embeddings) работает лучше всего в среднем по всем моделям? Как думаете, почему?
3. Насколько все модели превосходят DummyClassifier? О чём это говорит?
4. Есть ли модель, которая стабильно хуже остальных на всех типах признаков? Как думаете, почему?

YOUR ANSWER HERE

### Репорт результатов

В этом ДЗ можно получить максимум 9 баллов. Оценку 10 могут получить первые 15 человек потока, набравшие максимальное значение на тесте. Для этого необходимо в конце ДЗ прислать свое ФИО и лучший результат на тесте в [форму](https://forms.gle/Sk3NbUzNi4rqDpLj8).

### Интерпретируемость модели

Линейные модели (LogReg, SVM) удобны тем, что их коэффициенты можно проинтерпретировать: большой положительный коэффициент у слова означает, что это слово сильно говорит в пользу данного класса.

Ниже мы выведем самые значимые слова для каждого класса (по модели LogReg на TF-IDF word), а также покажем примеры ошибок модели.

In [ ]:
print('=== Топ-10 наиболее значимых слов для каждого класса (LogReg + TF-IDF word) ===\n')

best_logreg = logreg_results['TF-IDF (word)']
lr_model = best_logreg.best_estimator_ if hasattr(best_logreg, 'best_estimator_') else best_logreg
feat_names = tfidf_w_vec.get_feature_names_out()

for i, cls_name in enumerate(le.classes_):
    coefs = lr_model.coef_[i]
    top_idx = np.argsort(coefs)[-10:][::-1]
    top_words = [feat_names[j] for j in top_idx]
    print(f'{cls_name}: {", ".join(top_words)}')

#### Задание 7.3. Интерпретация модели (0.75 балла)
Проанализируйте результаты интерпретируемости модели и выпишите ответы **в следующей ячейке**:

1. Посмотрите на топ-10 значимых слов для нескольких классов. Насколько они интуитивно понятны? Соответствуют ли они тому, что вы ожидали для текстов этих авторов?
2. Рассмотрите примеры ошибок модели. Можете ли вы понять, почему модель ошиблась в этих случаях? Есть ли общий паттерн в ошибках?
3. Какие выводы можно сделать о том, чему модель «научилась»? На какие паттерны в тексте она опирается при классификации?

YOUR ANSWER HERE